[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/06-agentic-ai/03-mbox_as_an_mcp_server.ipynb)

In [1]:
# !pip install mbox mcp

# M|BOX as an MCP Server

The previous notebook wired an M|BOX search into the OpenAI API: a hand-written JSON schema, a hand-written function, and a hand-written loop to run the tool call and send the result back. That integration has to be repeated for every provider, OpenAI's schema is not Anthropic's, which is not Google's. The Model Context Protocol (MCP) exists to remove that repetition: write the tool once, as a standard MCP server, and any MCP-capable client, Claude Desktop, Claude Code, or your own agent, can use it without provider-specific glue.

In this notebook you will:

1. Write the same kind of M|BOX-backed search as a standalone MCP server
2. Connect to it the way a real MCP client does, over a subprocess and stdio, not an in-process import
3. Inspect the tool schema MCP derived automatically from the function signature, and compare it to the schema we wrote by hand in `02`
4. Call the tool for real and get back the same structured result

> Note: this notebook spawns `mcp_server.py` as a separate process and talks to it over stdio, the way an MCP client actually does. Run the cells in order.

## 1. The server

`%%writefile` below writes out `mcp_server.py` into this directory. It is a real, standalone script, meant to be run as its own process, not imported. The tool itself, `search_products`, is the same logic as the OpenAI function from `02`: fuzzy name matching, an optional price constraint via `NUM_LOWER`, and a fallback search that reports the closest match even when it's excluded by budget.

One detail is worth calling out before you read past it: MCP servers talk to their client over stdio, which means **stdout is reserved for protocol messages only**. M|BOX logs a license line and a startup banner to stdout as soon as it's imported, so the script redirects stdout to the null device around the import and index build to keep that off the wire.

In [2]:
%%writefile mcp_server.py
"""An MCP server exposing an M|BOX-backed product search as a single standard tool.

MCP over stdio uses stdout exclusively for JSON-RPC protocol messages, so anything
else written to it corrupts the transport. M|BOX itself logs a license line and a
startup banner as soon as it's imported, so we redirect stdout to the null device
for the import and index build to keep those off the wire. This catches the
synchronous log line reliably; see the notebook for a caveat on the rest.
"""
import os
import time

_stdout_fd = os.dup(1)
_devnull_fd = os.open(os.devnull, os.O_WRONLY)
os.dup2(_devnull_fd, 1)
try:
    import pandas as pd
    from mbox.indexing import TableIndexer
    from mbox.recall import TableRecallConfig, TableRecallFieldConfig, TableRecallMode

    catalog = pd.read_csv("datasets/product_catalog.csv")
    index = TableIndexer.create_index(
        catalog, index_columns=["product_name", "description", "unit_price"], tmp_dir="tmp_index_mcp"
    )
    time.sleep(2)
finally:
    os.dup2(_stdout_fd, 1)
    os.close(_stdout_fd)
    os.close(_devnull_fd)

from mcp.server import MCPServer

server = MCPServer("mbox-catalog")


def _text_only_match(product_name: str, max_results: int) -> pd.DataFrame:
    config = TableRecallConfig(
        fields=[TableRecallFieldConfig(input_column="product_name", indexed_column="product_name",
                                        minimum_quality=0, weight=100, mode=TableRecallMode.APPROX)],
        max_results=max_results, min_total_match_value=0, include_field_scores=True
    )
    return index.match(queries=pd.DataFrame({"product_name": [product_name]}), config=config)


@server.tool()
def search_products(product_name: str, max_price: float | None = None, max_results: int = 3) -> dict:
    """Search the product catalog by name, tolerating typos, optionally constrained by a maximum price."""
    if max_price is None:
        r = _text_only_match(product_name, max_results)
        if len(r) == 0 or r["index_row"].iloc[0] == -1:
            return {"found": False, "matches": []}
        matches = [{"product_name": row["product_name_candidate"],
                    "match_confidence": int(row["product_name_score"]),
                    "overall_score": int(row["overall_score"])} for _, row in r.iterrows()]
        return {"found": True, "matches": matches}

    fields = [
        TableRecallFieldConfig(input_column="product_name", indexed_column="product_name",
                                minimum_quality=0, weight=70, mode=TableRecallMode.APPROX),
        TableRecallFieldConfig(input_column="unit_price", indexed_column="unit_price",
                                minimum_quality=0, weight=30, mode=TableRecallMode.NUM_LOWER)
    ]
    config = TableRecallConfig(fields=fields, max_results=max_results, min_total_match_value=0, include_field_scores=True)
    r = index.match(queries=pd.DataFrame({"product_name": [product_name], "unit_price": [max_price]}), config=config)

    if len(r) > 0 and r["index_row"].iloc[0] != -1:
        matches = [{"product_name": row["product_name_candidate"],
                    "match_confidence": int(row["product_name_score"]),
                    "overall_score": int(row["overall_score"])} for _, row in r.iterrows()]
        return {"found": True, "matches": matches}

    unconstrained = _text_only_match(product_name, max_results=1)
    if len(unconstrained) > 0 and unconstrained["index_row"].iloc[0] != -1:
        row = unconstrained.iloc[0]
        actual_price = catalog.loc[catalog["product_name"] == row["product_name_candidate"], "unit_price"].iloc[0]
        return {
            "found": False,
            "matches": [],
            "note": "A matching product exists but exceeds the requested max_price.",
            "closest_match": {
                "product_name": row["product_name_candidate"],
                "match_confidence": int(row["product_name_score"]),
                "actual_price": float(actual_price)
            }
        }

    return {"found": False, "matches": []}


if __name__ == "__main__":
    server.run()


Overwriting mcp_server.py


## 2. Connect like a real client would

A real MCP client does not import your tool's code. It launches your server as a subprocess and speaks JSON-RPC to it over stdin/stdout. `stdio_client` below does exactly that: it starts `python mcp_server.py`, opens a session, and asks the server what tools it has.

> **Known issue, being fixed upstream:** part of M|BOX's startup banner is emitted in a way that can slip past the stdout redirect in `mcp_server.py` on some platforms and land on the stdio transport. The MCP client tolerates this, it skips the malformed line and logs a warning rather than failing the session, but that warning is noisy enough to be distracting in a notebook. The cell below quiets that specific logger as a workaround; M|BOX itself is being changed so this redirect isn't necessary at all in an upcoming release.

In [3]:
import logging

# Workaround for a known M|BOX issue (fix planned upstream): a fragment of its
# startup banner can bypass the stdout redirect in mcp_server.py and reach the
# MCP transport. The client already skips the resulting malformed line safely,
# this just quiets the warning it logs when that happens.
logging.getLogger("mcp.client.stdio").setLevel(logging.CRITICAL)

In [4]:
import sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(command="python", args=["mcp_server.py"])

async with stdio_client(server_params, errlog=sys.__stderr__) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        tools = await session.list_tools()
        tool = tools.tools[0]
        print(f"Tool: {tool.name}\n")
        print(f"Description: {tool.description}\n")
        import json
        print("Auto-derived input schema:")
        print(json.dumps(tool.input_schema, indent=2))

Tool: search_products

Description: Search the product catalog by name, tolerating typos, optionally constrained by a maximum price.

Auto-derived input schema:
{
  "properties": {
    "product_name": {
      "title": "Product Name",
      "type": "string"
    },
    "max_price": {
      "anyOf": [
        {
          "type": "number"
        },
        {
          "type": "null"
        }
      ],
      "default": null,
      "title": "Max Price"
    },
    "max_results": {
      "default": 3,
      "title": "Max Results",
      "type": "integer"
    }
  },
  "required": [
    "product_name"
  ],
  "type": "object",
  "title": "search_productsArguments"
}


Compare that schema to the one we wrote by hand in `02-mbox_as_an_openai_function.ipynb`. There, every property, its type, and its description were typed out manually in a Python dict. Here, MCP derived the same information, parameter names, types, defaults, and which are required, directly from the function's type hints. The one thing MCP could not infer on its own is the tool's *description*, which is why it's worth writing a docstring that actually explains what the tool does and how to interpret its output, the same way the description in `02` spelled out what `match_confidence` and `closest_match` mean. MCP automates the schema, not the judgment calls about how to use the tool.

## 3. Call the tool for real

A typo'd product name, with a budget wide enough to actually include the real match this time, so you can see what a confident, successful result looks like end to end. The session below opens a fresh connection, calls `search_products`, and prints back exactly what the server returned, structured data, not prose.

In [5]:
import json

async with stdio_client(server_params, errlog=sys.__stderr__) as (read, write):
    async with ClientSession(read, write) as session:
        await session.initialize()
        result = await session.call_tool(
            "search_products", {"product_name": "extendd batery pak", "max_price": 30}
        )
        payload = json.loads(result.content[0].text)
        print(json.dumps(payload, indent=2))

{
  "found": true,
  "matches": [
    {
      "product_name": "Extended Battery Pack",
      "match_confidence": 60,
      "overall_score": 72
    }
  ]
}


A confident match, `Extended Battery Pack`, well within the `$30` budget, with a `match_confidence` of `60` on the name alone and an `overall_score` of `72` once the price constraint factors in, all despite two typos in the query. Same result shape as the OpenAI-function version in `02`, because it is the same M|BOX call underneath. What changed is not the search logic, it's that the exact same server process would answer this identically for a Claude Desktop conversation, a Claude Code session, or any other MCP client, with zero provider-specific code.

## 4. Practical notes

**Keep noisy dependencies out of stdout.** Anything a library prints to stdout while an MCP stdio server is running is a potential protocol corruption, not just visual noise. M|BOX's own startup banner is the example in this notebook, redirecting stdout around the import catches most of it, and an upcoming M|BOX release removes the need for that redirect entirely, but until then, the client's tolerance for an occasional malformed line, skip it and log a warning rather than fail the session, is what kept this notebook's tool calls working end to end regardless. Test what your own dependencies print before wiring them into a stdio server, and don't assume every MCP client is this forgiving.

**The schema is only as good as your type hints and docstring.** MCP will happily generate a schema from untyped parameters, but the result will be far less useful to whatever model is deciding how to call your tool. Type everything, and write the docstring the way you would write a schema description by hand.

**One server, many tools.** This notebook exposed a single `search_products` function, but `@server.tool()` stacks: a real deployment would expose several related M|BOX searches, over different tables or with different default recall configs, from the same server process, each independently discoverable and callable.

**stdio is one transport among several.** `server.run()` defaults to `stdio`, which is what a locally-launched client like Claude Desktop expects. The same server can run over `streamable-http` for a remotely hosted tool instead, without changing the tool code itself.

## Next steps

- **`04-mcp_server_advanced.ipynb`** - build the index once and reuse it across several search endpoints in the same server
- **`05-grounding_rag_with_deterministic_matching.ipynb`** - apply this same grounding principle to retrieval-augmented generation